# TargetGym quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YannBerthelot/TargetGym/blob/main/notebooks/quickstart.ipynb)

**TargetGym** is a set of JAX environments for *target MDPs*: tasks where you
reach a setpoint and then **hold it**, instead of reaching a goal once and
stopping. That is what most industrial control looks like.

This notebook runs on a free CPU runtime in about a minute. It covers:

1. running the tuned controller that ships with every environment
2. seeing what it scores, so you know what a learned policy has to beat
3. stepping 256 environments at once under `jit`, `vmap` and `scan`
4. using the Gymnasium interface with your existing tools

Docs: <https://yannberthelot.github.io/TargetGym/>

In [ ]:
!pip install -q target-gym

In [ ]:
import time

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import target_gym as tg

print(f"target-gym {tg.__version__}")
names = tg.list_envs()
print(f"{len(names)} tasks: {', '.join(names)}")

## 1. Every environment ships an expert

A benchmark whose only reference point is a random policy tells you an agent
learned *something*. One with the controller practitioners deploy tells you
whether it learned anything **useful**. Here is the 2D aircraft holding a
commanded altitude, flown by its expert.

In [ ]:
env, params = tg.make("plane", max_steps_in_episode=600)

pid = tg.make_expert("plane").host()   # keeps its own memory
pid.reset()                             # at the start of every episode

obs, state = env.reset(jax.random.PRNGKey(0), params)

altitude, target = [], []
for t in range(int(params.max_steps_in_episode)):
    # Record *before* stepping: see the note below on auto-reset.
    altitude.append(float(state.z))
    target.append(float(state.target_altitude))

    action = pid(np.asarray(obs))
    obs, state, reward, terminated, truncated, info = env.step(
        jax.random.PRNGKey(t), state, action, params
    )
    if terminated or truncated:
        break

print(f"{len(altitude)} steps, final error {abs(altitude[-1] - target[-1]):.1f} m")

plt.figure(figsize=(9, 3.5))
plt.plot(altitude, label="altitude")
plt.plot(target, "--", label="commanded")
plt.xlabel("step")
plt.ylabel("altitude (m)")
plt.title("plane, flown by its expert")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

> **One gotcha worth knowing.** `env.step` follows the gymnax convention and
> **auto-resets** when an episode ends, so the state it hands back after a
> termination already belongs to the *next* episode. Record what you care
> about before you step, as above, or read `info['final_observation']`. Reading
> `state` after the loop gives you a fresh episode's altitude and a fresh
> target, which looks like a controller that suddenly stopped working.
>
> `terminated` means the plant reached a terminal state, such as a crash.
> `truncated` means the time limit ran out. Only `terminated` should zero your
> value bootstrap.

## 2. What does a policy have to beat?

Every task is scored the same way: each step costs how far the tracked
quantities are from their targets, in units of the instrument's resolution, so
rewards are never positive and a smaller cost is better. `tg.score_policy` runs
a policy and the expert on the same seeds of the scored configuration and
compares them. Here a constant action on `reactor`, a power reactor's control-rod loop:

In [ ]:
env, params = tg.make("reactor")
action_shape = env.action_space(params).shape


def make_policy():                          # a fresh policy for each episode
    return lambda obs: np.zeros(action_shape)  # stand-in: put yours here


m = tg.score_policy("reactor", make_policy, seeds=3)   # 10 for a result
print(f"cost ratio vs the expert  {m['ratio']:.3f}  (above 1 beats it)")
print(f"seeds won                 {m['win_rate']:.0%}")
print(f"score                     {m['score']:.2f}  (0 = expert, 1 = ceiling)")
print(f"room beyond the expert    {m['headroom']:.1f} decades")

The ratio is the expert's cost over the policy's, so above 1 beats the expert.
The score runs from 0 at the expert to 1 at the task's ceiling, where every
tracked quantity sits within its instrument's resolution, and is negative below
the expert. The headroom is how far the ceiling lies beyond the expert: on some
tasks the expert already holds within the resolution, and only the moves
between targets leave room. `tg.make_oracle(name)`
gives the full-state MPC, which reads the true state and so shows what a
perfect model could do; nine tasks plan it with CasADi, which
`pip install 'target-gym[mpc]'` adds.

## 3. It is JAX the whole way down

The environments are written so a rollout never leaves the accelerator. Here we
step 256 of them in parallel, 400 steps deep, inside a single compiled `scan`.

In [ ]:
env, params = tg.make("cstr")

N_ENVS, HORIZON = 256, 400
action_shape = env.action_space(params).shape or (1,)


def episode(key):
    keys = jax.random.split(key, HORIZON + 1)
    _, state = env.reset(keys[0], params)

    def step(state, k):
        _, state, reward, _, _, _ = env.step(
            k, state, jnp.zeros(action_shape), params
        )
        return state, reward

    _, rewards = jax.lax.scan(step, state, keys[1:])
    return rewards.sum()


batch = jax.jit(jax.vmap(episode))
keys = jax.random.split(jax.random.PRNGKey(0), N_ENVS)

batch(keys).block_until_ready()  # compile once, then measure
start = time.perf_counter()
returns = batch(keys).block_until_ready()
elapsed = time.perf_counter() - start

steps = N_ENVS * HORIZON
print(f"{steps:,} steps in {elapsed:.3f} s = {steps / elapsed / 1e6:.1f} M steps/s")
print(f"mean return {float(returns.mean()):.1f} over {N_ENVS} environments")

Throughput varies a lot across the suite, from a first-order lag to a kiln with
a half-hour transport delay, which is deliberate: you can iterate quickly on the
cheap ones and spend the sample budget where the dynamics are actually hard.

## 4. Or use it through Gymnasium

If your tooling expects Gymnasium, there is a wrapper, so stable-baselines3 and
friends work unchanged. You lose end-to-end GPU execution, since the agent then
lives outside JAX, but nothing else changes.

In [ ]:
gym_env = tg.make_gymnasium("plane")
obs, info = gym_env.reset(seed=0)

for _ in range(5):
    obs, reward, terminated, truncated, info = gym_env.step(
        gym_env.action_space.sample()
    )

print(f"observation {obs.shape}, reward {float(reward):.3f}")
print(f"action space {gym_env.action_space}")

## Where to go next

- **[Choosing an environment](https://yannberthelot.github.io/TargetGym/latest/complexity/)**
  and the **[environment reference](https://yannberthelot.github.io/TargetGym/latest/environments/)**
  for shapes, tracked variables and the physics contract behind each task.
- **[Using the PIDs](https://yannberthelot.github.io/TargetGym/latest/using-the-pids/)**
  for evaluating against and training on the experts.
- **[RL protocol](https://yannberthelot.github.io/TargetGym/latest/rl-protocol/)**
  for how a result is scored and reported.

When you publish a number, cite the **versioned** task name, `plane-v2` rather
than `plane`. The version changes when the dynamics, the reward, the
parameters or the observation layout change, so a result quoted against it
keeps meaning what it meant.